# Generate Cluster Table

Bottom-up construction of the TESS cluster data table.

**Steps:**
1. Build base table from Wainer 2023 catalogs (name, age, origin)
2. Expand into one row per distinct sector combination; compute LC + LSP per row
3. Add variability summary statistics
4. (Follow-up) Add GP information
5. Save table to disk

In [ ]:
# import os

# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
print('hi')

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys, os

# Ensure GenerateTable_CGW is on the path so sub-packages resolve
MODULE_DIR = os.path.dirname(os.path.abspath('generate_table.ipynb'))
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# DATA_DIR = '../data'
# LC_DIR   = '../light_curves'
# INTERMEDIATE_TABLE_NAME   = './data/intermediate_table_4sigma'
# #OUTPUT   = './data/cluster_table_3sectorcombos_per_cluster_SHOQprior_RNperiodCut'
# OUTPUT   = './data/cluster_table_3sectorcombos_per_cluster_test'


DATA_DIR = '../data'
LC_DIR   = '../light_curves_synthetic'
INTERMEDIATE_TABLE_NAME   = './data/intermediate_table_Aug21_synthetic'
#OUTPUT   = './data/cluster_table_3sectorcombos_per_cluster_SHOQprior_RNperiodCut'
OUTPUT   = './data/cluster_table_Aug21_3sectorcombos_synthetic'



In [ ]:
print('MODULE_DIR: ', MODULE_DIR)
print('DATA_DIR  : ', DATA_DIR)
print('LC_DIR    : ', LC_DIR)
print('INTERMEDIATE_TABLE_NAME    : ', INTERMEDIATE_TABLE_NAME)
print('OUTPUT    : ', OUTPUT)

In [ ]:
from table_pipeline.table_io import save_as_fits, load_from_fits, save_as_pickle, load_from_pickle, save_as_hdf5, load_from_hdf5
#table = load_from_fits(OUTPUT)

## Step 1 — Base table (name / age / origin)

In [ ]:
from table_pipeline.wainer_table import build_base_table

base_table = build_base_table(DATA_DIR)

## --- optional subset for quick runs -------------------------------------------------
## Filtering base_table here propagates through the ENTIRE pipeline with no other
## changes: expand_table just iterates these rows, and `origin` is only ever used to
## pick the MW/LMC/SMC subdirectory in get_lc_path.
## It is also upstream of LC_DIR, so the same line subsets a real run and a synthetic
## run identically -- which is what makes the two directly comparable.
ORIGINS = ['MW']                                                      # 124 clusters
base_table = base_table[base_table['origin'].isin(ORIGINS)].reset_index(drop=True)
## ------------------------------------------------------------------------------------

print(f"Total clusters: {len(base_table)}")
base_table.head()

## Steps 2 / 2.5 / 2.75 — Expand by sector combos + LC + LSP

For each cluster:
- Discovers sectors in the FITS file
- Groups by cadence (no combo mixes different cadences)
- Generates distinct sector combinations via `make_distinct_sector_combinations`
- Loads, resamples, and normalizes the LC for each combo
- Computes LSP per combo

**This step is slow** — runs through all clusters. Adjust `f_max` to control combo density.

In [ ]:
%%time
from table_pipeline.sector_expansion import expand_table

full_table = expand_table(
    base_table,
    lc_dir=LC_DIR,
    f_max=0.7,                #1 is minimum combos short, 0 is exhaustive combos
    seed=0,
    shuffle=False,
    include_full=True,

    P_min=(30/60/24)*4, 
    P_max=10.0,
    
    n_wn_boot=200, #Computes the WN threshold in addition to FAP for use in the GP fitting step.
    wn_percentile=99.0,
    alpha=10,

    nsigma=4, #for mad sigma clipping

    n_workers=32,
    
 )

print(f"Expanded table: {len(full_table)} rows")
print(f"Columns: {list(full_table.columns)}")
full_table[['name', 'age', 'origin', 'sectors', 'n_sectors', 'cadence']].head(10)

In [ ]:
full_table

### Save/Load Table

In [ ]:
save_as_pickle(full_table, INTERMEDIATE_TABLE_NAME+'.pkl')

In [ ]:
%%time
full_table = load_from_pickle(INTERMEDIATE_TABLE_NAME+'.pkl')

### Check Table

In [ ]:
type(full_table.iloc[0]['sectors'])

In [ ]:
i = full_table['name']=='[SL63] 106'
full_table[i][['name', 'age', 'origin', 'sectors', 'n_sectors', 'cadence']].head(40)

In [ ]:
len(full_table['name'].unique())

In [ ]:
# Full Table
#table = copy.deepcopy(full_table)

##### %%%--------------------------------------------------------------------------------%%% ######

# Only sector per cluster 
#filtered = full_table[full_table['n_sectors'] == 1]
#table = filtered.groupby('name').sample(1)

##### %%%--------------------------------------------------------------------------------%%% ######

# Only sector of a subset of clusters
# filtered = full_table[full_table['n_sectors'] == 1]
# filtered = filtered.groupby('name').sample(1)
# table = filtered.sample(n=10)

##### %%%--------------------------------------------------------------------------------%%% ######

# Only sector of a subset of clusters which includes the high-variability ones
# filtered = full_table[full_table['n_sectors'] == 1]
# filtered = filtered.groupby('name').sample(1)
# must_have = ['BERKELEY 83', 'CZERNIK 13', 'KMHK1378']
# fixed = filtered[filtered['name'].isin(must_have)]
# rest = filtered[~filtered['name'].isin(must_have)].sample(n=7)
# table = pd.concat([fixed, rest])

##### %%%--------------------------------------------------------------------------------%%% ######

## select 2 rows for each cluster, both rows being 1 sector data, but each of a different 'cadence'. 
## if no 2 cadences exist, use 2 of a single cadence. BUT only select a subset of 'other' clusters.
# filtered = full_table[full_table['n_sectors'] == 1]
# rows = []
# for name, group in filtered.groupby('name'):
#     cadences = group['cadence'].unique()
#     if len(cadences) >= 2:
#         for c in cadences[:2]:
#             rows.append(group[group['cadence'] == c].sample(1))
#     else:
#         rows.append(group.sample(n=min(2, len(group))))
# paired = pd.concat(rows).reset_index(drop=True)
# print('name' in paired.columns)
# must_have = ['BERKELEY 83', 'CZERNIK 13', 'KMHK1378']
# fixed = paired[paired['name'].isin(must_have)]
# rest_names = paired[~paired['name'].isin(must_have)]['name'].unique()
# rest_names = np.random.choice(rest_names, size=len(paired)-3, replace=False)
# rest = paired[paired['name'].isin(rest_names)]
# table = pd.concat([fixed, rest]).reset_index(drop=True)

##### %%%--------------------------------------------------------------------------------%%% ######

## select 2 rows for each cluster, both rows being 1 sector data, but each of a different 'cadence'. 
## if no 2 cadences exist, use 2 of a single cadence
# filtered = full_table[full_table['n_sectors'] == 1]
# rows = []
# for name, group in filtered.groupby('name'):
#     cadences = group['cadence'].unique()
#     if len(cadences) >= 2:
#         for c in cadences[:2]:
#             rows.append(group[group['cadence'] == c].sample(1))
#     else:
#         rows.append(group.sample(n=min(2, len(group))))
# table = pd.concat(rows).reset_index(drop=True)


##### %%%--------------------------------------------------------------------------------%%% ######


## select 3 secotrs for each cluster, all rows being 1 sector data, and then also save the rows with their combos. all of the same 'cadence'. 
## if no 2 cadences exist, use 2 of a single cadence
from itertools import combinations

# Seeded so a real run and a synthetic run select the SAME sector combos -- otherwise
# the two tables are not row-matched and the difference reads as a pipeline effect.
RNG_SEED = 0

rows = []
for name, cluster in full_table.groupby('name'):
    multi = cluster[cluster['n_sectors'] > 1]
    cad = (multi if len(multi) else cluster)['cadence'].mode()[0]
    sub = cluster[cluster['cadence'] == cad]
    keys = sub['sectors'].map(tuple)   # lists aren't hashable/comparable; tuples are

    triples = sub[sub['n_sectors'] == 3]
    if len(triples):
        secs = list(triples.sample(1, random_state=RNG_SEED)['sectors'].iloc[0])
    else:
        secs = (sub.loc[sub['n_sectors'] == 1, 'sectors']
                .map(lambda s: s[0]).drop_duplicates()
                .sample(frac=1, random_state=RNG_SEED).head(3).tolist())

    for r in (1, 2, 3):
        for combo in combinations(sorted(secs), r):
            rows.append(sub[keys.isin([combo])].head(1))
table = pd.concat(rows).reset_index(drop=True)


print('final table length', len(table))
table

In [ ]:
np.unique(table['name'])

In [ ]:
row = table.iloc[0]

print(row['rn_log10_N'])
print(row['rn_alpha'])
print(1/row['LSP_freq'][np.argmax(row['LSP_power'])]) 

In [ ]:
# = table['name']=='[SL63] 106'
i = table['name']=='ASCC 116'

table[i][['name', 'age', 'origin', 'sectors', 'n_sectors', 'cadence']].head(40)

In [ ]:
len(table)

In [ ]:

def plot_noise(row):
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(26, 4))
    
    ax1.errorbar(row['LC_t'], row['LC_flux'], row['LC_flux_err'], lw=0.3, ecolor='lightcoral')
    ax1.set(xlabel='time (days)', ylabel='normalized flux', title=f"{row['name']} — LC")
    
    freq  = row['LSP_freq']
    power = row['LSP_power']
    age = row['age']
    print('age', age)
    
    # rn_P_empirical / rn_P_theoretical are no longer stored -- P_N_theoretical was a
    # hardcoded 1.0 and P_N_empirical was constant to ~0.3%, and neither was used
    # downstream. Tolerate both old and new tables.
    log10_N, alpha = row['rn_log10_N'], row['rn_alpha']
    P_N_empirical = row['rn_P_empirical'] if 'rn_P_empirical' in row.index else None
    P_N_theoretical = 1.0
    P_model   = 10**log10_N * freq**(-alpha)
    n_freq    = len(freq)
    gamma     = -np.log(1 - 0.99**(1/n_freq))
    threshold = P_model * gamma
    
    for ax, xscale, yscale, title_suffix in [
      (ax2, 'linear', 'log',    'log-y'),
      (ax3, 'log',    'log',    'log-log'),
      (ax4, 'linear', 'linear', 'linear'),
    ]:
      #ax.plot(freq, power, lw=0.8, color='steelblue', zorder=2)
      ax.scatter(freq, power, s=0.5, color='steelblue', zorder=2)
      ax.axhline(row['LSP_FAP_power'], color='red', ls='--', lw=1,
                 label='1% FAP (analytic)')
      ax.axhline(row['LSP_WN_threshold'], color='darkorange', ls='--', lw=1,
                 label='WN threshold (scalar, 99th pct)')
      ax.plot(freq, P_model, color='firebrick', lw=1, ls='-.',
              label=f'Red-noise continuum (α={alpha:.2f})', zorder=3)
      ax.plot(freq, threshold, color='darkred', lw=1.2, ls='-',
              label='Red-noise threshold (FAP=1%)', zorder=3)
      if P_N_empirical is not None:
          ax.axhline(P_N_empirical, color='black', ls='--', lw=1,
                     label=f'P_N empirical ({P_N_empirical:.3f})')
      ax.axhline(P_N_theoretical, color='black', ls=':', lw=1,
                 label=f'P_N theoretical ({P_N_theoretical:.1f})')
      ax.set(xlabel='frequency (1/day)', ylabel='power',
             xscale=xscale, yscale=yscale,
             title=f"{row['name']} — LSP [{title_suffix}]  sectors={row['sectors']}")
      ax.legend(fontsize=7)
      if title_suffix == 'linear':
          ax.set_ylim(0, float(np.max(power)*1.5))
    
    plt.tight_layout()
    plt.show()



# Quick sanity check: plot the LC and LSP of the first row

# row = table.iloc[-1]

# plot_noise(row)

# #test_table = table.iloc[0:20]
# test_table = table[
#     (table["name"].isin(['BERKELEY 83', 'CZERNIK 13', 'KMHK1378'])) & 
#     (table["n_sectors"] == 1)     
# ]
# row = test_table.iloc[-1]

# plot_noise(row)


# for i in range(0, len(table), len(table)//60):
#     row = table.iloc[i]
#     plot_noise(row)
#     row = table.iloc[i+1]
#     plot_noise(row)
#     row = table.iloc[i+2]
#     plot_noise(row)
#     row = table.iloc[i+3]
#     plot_noise(row)
#     row = table.iloc[i+4]
#     plot_noise(row)
#     row = table.iloc[i+5]
#     plot_noise(row)
#     row = table.iloc[i+6]
#     plot_noise(row)

t = table[table['name']=='FSR 1712']
for i in range(0, len(t)):
    row = t.iloc[i]
    plot_noise(row)


In [ ]:
# Empirical noise floor from the actual LSP at high frequencies
freqs = row['LSP_freq']
power = row['LSP_power']

high_freq_mask = freqs > 5  # well above any astrophysical signal
empirical_floor = np.median(power[high_freq_mask])


In [ ]:
row = table[table['name']=='BERKELEY 83'].iloc[0]
t = row['LC_t']
flux = row['LC_flux']
err = row['LC_flux_err']
freqs = row['LSP_freq']

from astropy.timeseries import LombScargle

plt.figure(figsize=(10, 10))

configs = [
    ('psd, unweighted', 'psd', None),
    ('psd, weighted', 'psd', err),
    ('standard, unweighted', 'standard', None),
    ('standard, weighted', 'standard', err),
]

for label, norm, dy in configs:
    if dy is not None:
        ls = LombScargle(t, flux, dy, normalization=norm)
    else:
        ls = LombScargle(t, flux, normalization=norm)
    power = ls.power(freqs)
    print(f"{label} high-freq median: {np.median(power[freqs>7]):.6f}")
    line, = plt.plot(freqs, power, label=label)

    fap_baluev = ls.false_alarm_level(0.01, method='baluev')
    fap_bootstrap = ls.false_alarm_level(0.01, method='bootstrap', method_kwds={'n_bootstraps': 1000})
    print(f"  Baluev: {fap_baluev:.6e}, Bootstrap: {fap_bootstrap:.6e}")

    color = line.get_color()
    plt.axhline(fap_baluev, color=color, linestyle='--', alpha=0.5, label=f'{label} FAP (baluev)')
    plt.axhline(fap_bootstrap, color=color, linestyle=':', alpha=0.5, label=f'{label} FAP (bootstrap)')

plt.yscale('log')
plt.legend()

In [ ]:
P_N_theory-P_N_true

In [ ]:
print('hi')

## Step 3 — Summary statistics

In [ ]:
%%time 
from table_pipeline.summary_stats import add_variability_metrics

add_variability_metrics(table)

stat_cols = ['intrinsic_std', 'intrinsic_rms', 'intrinsic_mad',
             'n_bins_used', 'vn_ratio_gap_aware', 'stetson_j_gap_aware',
             'mean_median_offset', 'gamma_p',
             'sigma_mad', 'excess_var', 'snr']
table[stat_cols].describe()

In [ ]:
%%time
# Step 3b — per-sector shape statistics
# gamma_p and g1_int are computed per sector and n-weighted averaged, NOT on the
# stitched array: the pooled versions reweight sectors by sigma_k (gamma_p) and
# sigma_k^3 (g1), so a stitched skewness drifts systematically with n_sectors.
# The amplitudes (excess_var, sigma_mad, mean_median_offset) stay pooled — they mix
# linearly and need no such treatment.
from table_pipeline.sector_stats import add_sector_stats

add_sector_stats(table, lc_dir=LC_DIR, snr_min=1.0)

sec_cols = ['gamma_p_sec', 'g1_int_sec', 'g1_valid', 'n_sectors_valid']
print(f"g1_valid: {int(table['g1_valid'].sum())}/{len(table)} rows")
table[['excess_var', 'snr', 'sigma_mad', 'mean_median_offset',
       'gamma_p_sec', 'g1_int_sec', 'n_sectors']].describe()


In [ ]:
%%time
# Step 3c — LSP peak statistics (post-hoc; no expand_table rerun needed)
# Restores the peak power/period that compute_lsp computes but sector_expansion
# discarded, plus the red-noise-normalised significance.
#
# Locate with the difference, test with the ratio:
#   LSP_peak_period_diff = argmax(P - C)  -- the UNBIASED locator (E[P-C] = S, flat in
#                                            period); use this as the rotation candidate
#   LSP_peak_period      = argmax(P)      -- biased toward long periods
#   LSP_peak_snr_rn      = max(P/(C*g))   -- global detection, significant when > 1
#   LSP_peak_snr_at_diff = P/(C*g) at the located peak -- gate the period on THIS
from table_pipeline.lsp_stats import add_lsp_metrics, check_lsp_harmonics

add_lsp_metrics(table)

table[['LSP_peak_power', 'LSP_peak_period', 'LSP_peak_snr_rn',
       'LSP_peak_period_diff', 'LSP_peak_excess_diff',
       'LSP_peak_snr_at_diff']].describe()


In [ ]:
# Harmonic / systematics check — run standalone any time after add_lsp_metrics.
# excess_ratio >> 1 at a suspect period means peaks are piling up on a TESS
# systematic (momentum dumps, scattered light) rather than on real rotation.
harm = check_lsp_harmonics(table, tol=0.02)


## Step 4 — GP pipeline

Fit a multi-component SHO Gaussian Process to each row using the pre-computed LSP.
Results are stored in the table as `gp_result` (GPFitResult object) plus derived
columns for spectral statistics.

In [ ]:
%%time
from gp_pipeline.gp_table_io import add_gp_fits

add_gp_fits(
    table,
    save_path="gp_fit_data_test2.pkl",
    freq_lim=(0.1, 12.0),   # 1/day; matches LSP_freq range
    n_bins=10, # number of bins in gp summary stats
    
    max_components=10,
    
    Q_min=0.5,
    Q_max=500.0,
    init_Q=20.0,    
    
    fit_full_only=True,              # set to True If you only want to fit the full n_peaks SHO components version of the model.
    debug_fitalln=True,              # True turns off early BIC stopping
    bic_improvement_threshold=10.0,  # How much BIC needs to improve by before early stopping kicks in

    threshold_mode='red_noise',
    # threshold_mode='red_noise_subtracted', 
    # excess_k=0.0 for the pure LSP-RN version, or set to one for the full signicane version (ie same as 'red_noise') full fractions inbetween.
    
    harmonic_tolerance=0.05,
    harmonic_masking=False,

    two_phase_fitting=True,
    use_lognormal_period_prior=True,
    fit_red_noise_component=True,

    use_q_prior=True, q_prior_weight=1.0,  #Discoruages the SHO terms from becoming RN terms. Higher q_prior_weight means stronger penality.
    rn_period_cap=10.0, #Caps the period of the RN term (prevents multisectior 100+ day period shinanigans).

    
    verbose=0,
)

print("GP columns added:", [c for c in table.columns if c.startswith("gp") or c.startswith("GP")])

In [ ]:
print("GP columns added:", [c for c in table.columns if c.startswith("gp") or c.startswith("GP")])

In [ ]:
print('ho')

In [ ]:
from gp_pipeline.gp_table_io import load_gp_results
from gp_pipeline.visualize_gp_afterfit import plot_gp_fit, plot_kspace, plot_noise_gp

table_subset = table.sample(n=10)

for i in range(0, len(table_subset)):
    row = table_subset.iloc[i]
    result = row["gp_result"]
    if result is not None:
        print(f"n_components={result.n_components}  BIC={result.bic:.1f}  periods={result.periods}")
    else:
        print("No GP result for this row")

plot_gp_fit(table_subset)
plot_kspace(table_subset, log_y=True)
plot_noise_gp(table_subset) 

In [ ]:
table_subset.columns

In [ ]:
r = table_subset.iloc[6]["gp_result"]
print(r)
print(r.features)


In [ ]:
from gp_pipeline.gp_table_io import load_gp_results
from gp_pipeline.visualize_gp_afterfit import plot_gp_fit, plot_kspace, plot_noise_gp

#table_subset = table[table['name']=='FSR 1769']
#table_subset = table[table['name']=='ASCC 116']
table_subset = table


for i in range(0, len(table_subset)):
    row = table_subset.iloc[i]
    result = row["gp_result"]
    if result is not None:
        print(f"n_components={result.n_components}  BIC={result.bic:.1f}  periods={result.periods}")
    else:
        print("No GP result for this row")

#plot_gp_fit(table_subset)
#plot_kspace(table_subset, log_y=True)
plot_noise_gp(table_subset, save_path='/astro/users/cgwill/TESS_Cluster_Age_ML/GenerateTable_CGW/figs/3sectors_Aug21_synthetic') 

In [ ]:
pwd

In [ ]:
from gp_pipeline.gp_table_io import load_gp_results
from gp_pipeline.visualize_gp_afterfit import plot_gp_fit, plot_kspace, plot_noise_gp

#table_subset = table[table['name']=='FSR 1769']
#table_subset = table[table['name']=='ASCC 116']
table_subset = table


for i in range(0, len(table_subset)):
    row = table_subset.iloc[i]
    result = row["gp_result"]
    if result is not None:
        print(f"n_components={result.n_components}  BIC={result.bic:.1f}  periods={result.periods}")
    else:
        print("No GP result for this row")

#plot_gp_fit(table_subset)
#plot_kspace(table_subset, log_y=True)
plot_noise_gp(table_subset) 

In [ ]:
table.columns

In [ ]:
print('hi')

In [ ]:
i = table['name']=='BSDL2934'
y = table[i].iloc[0]['LC_flux']
yerr = table[i].iloc[0]['LC_flux_err']
jitter = table[i].iloc[0]['gp_jitter']

np.nanstd(y), np.nanmedian(yerr)

print('data variance', np.nanstd(y)**2)
print('instrament variance', np.nanmedian(yerr)**2)
print('jitter', jitter)
print(np.nanstd(y)/np.nanmedian(yerr))


In [ ]:
from gp_pipeline.gp_table_io import load_gp_results
from gp_pipeline.visualize_gp_afterfit import plot_gp_fit, plot_kspace, plot_noise_gp

#table_subset = table[table['name']=='FSR 1769']
#table_subset = table[table['name']=='ASCC 116']
table_subset = table


for i in range(0, len(table_subset)):
    row = table_subset.iloc[i]
    result = row["gp_result"]
    if result is not None:
        print(f"n_components={result.n_components}  BIC={result.bic:.1f}  periods={result.periods}")
    else:
        print("No GP result for this row")

#plot_gp_fit(table_subset)
#plot_kspace(table_subset, log_y=True)
plot_noise_gp(table_subset) 

In [ ]:
table

In [ ]:
table.columns

In [ ]:
med_err = table["LC_flux_err"].apply(np.median)
max_power = table["gp_kspace_log_band_powers"].apply(np.max)
cadence = table["cadence"]



fig, axs = plt.subplots(2, figsize=(10,20))
axs[0].scatter(med_err, table["cadence"], c=max_power)
axs[0].set_xscale('log')
axs[0].set_ylabel('cadence')
axs[0].set_xlabel('med_err')

axs[1].scatter(table['intrinsic_std']+1e-8, table['rn_alpha'], c=max_power)
axs[1].set_xscale('log')
axs[1].set_ylabel('rn_alpha')
axs[1].set_xlabel('log intrinsic_std')

axs[2].scatter(table['gp_kspace_log_low_high_ratio']+1e-8, table['rn_alpha'], c=max_power)
axs[2].set_xscale('log')
axs[2].set_ylabel('rn_alpha')
axs[2].set_xlabel('log intrinsic_std')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def is_vector_like(x):
    """Return True if x looks like a list/array of values rather than a scalar."""
    return isinstance(x, (list, tuple, np.ndarray, pd.Series))


def reduce_column_for_plot(series, reducer=np.nanmedian):
    """
    Convert a table column into a 1D numeric array for plotting.

    If entries are vectors/lists/arrays, reduce each row using `reducer`.
    If entries are scalars, return the column directly.
    """
    non_null = series.dropna()

    if len(non_null) == 0:
        return series

    example = non_null.iloc[0]

    if is_vector_like(example):
        return series.apply(lambda x: reducer(np.asarray(x, dtype=float)) if is_vector_like(x) else np.nan)

    return pd.to_numeric(series, errors="coerce")


# ----------------------------
# Define color variable
# ----------------------------
max_power = reduce_column_for_plot(table["gp_kspace_log_band_powers"], reducer=np.nanmax)

# ----------------------------
# Specify columns to plot
# ----------------------------
cols_to_plot = [
    "LC_flux_err",
    "cadence",
    "gp_bic",
    "intrinsic_std",
    "intrinsic_mad",
    'intrinsic_rms',
    'age',
    'n_sectors',
    'rn_alpha',
    'gp_kspace_log_low_high_ratio',

    
    # add more columns here
]


# Index(['name', 'age', 'origin', 'sectors', 'n_sectors', 'cadence', 'LC_t',
#        'LC_flux', 'LC_flux_err', 'LSP_freq', 'LSP_power', 'LSP_FAP_power',
#        'LSP_WN_threshold', 'rn_log10_N', 'rn_alpha', 'rn_P_empirical',
#        'rn_P_theoretical', 'intrinsic_std', 'intrinsic_rms', 'intrinsic_mad',
#        'n_bins_used', 'vn_ratio_gap_aware', 'stetson_j_gap_aware', 'gp_result',
#        'GP_freq', 'GP_PSD', 'GP_PSD_binned', 'GP_PSD_bin_ratios',
#        'GP_PSD_high_low_ratio', 'gp_n_components', 'gp_bic',
#        'gp_log_likelihood', 'gp_periods', 'gp_kspace_bin_edges',
#        'gp_kspace_bin_centers', 'gp_kspace_log_band_powers',
#        'gp_kspace_log_low_high_ratio', 'gp_kspace_log_band_ratios'],
#       dtype='str')

# Optional custom transforms for specific columns
def bic_transform(x):
    return np.log10(x - np.nanmin(x) + 1e-12)

special_transforms = {
    "gp_bic": bic_transform,
}

# Optional log x-axis columns
log_x_cols = [
    "LC_flux_err",
    # add more if desired
]


# ----------------------------
# Build plotting dataframe
# ----------------------------
plot_data = {}

for col in cols_to_plot:
    x = reduce_column_for_plot(table[col], reducer=np.nanmedian)

    if col in special_transforms:
        x = special_transforms[col](x)

    plot_data[col] = x

plot_df = pd.DataFrame(plot_data)
plot_df["max_power"] = max_power


# ----------------------------
# Plot each column vs max_power
# color = max_power
# ----------------------------
ncols = len(cols_to_plot)

fig, axs = plt.subplots(
    ncols,
    figsize=(10, 3.2 * ncols),
    sharey=False
)

if ncols == 1:
    axs = [axs]

for ax, col in zip(axs, cols_to_plot):
    sc = ax.scatter(
        plot_df[col],
        plot_df["max_power"],
        c=plot_df["max_power"],
        s=20,
        alpha=0.8
    )

    ax.set_xlabel(col)
    ax.set_ylabel("max gp_kspace_log_band_powers")
    ax.set_title(f"{col} vs max_power")

    if col in log_x_cols:
        ax.set_xscale("log")

    fig.colorbar(sc, ax=ax, label="max_power")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def is_vector_like(x):
    return isinstance(x, (list, tuple, np.ndarray, pd.Series))


def reduce_column_for_triangle(series, reducer=np.nanmedian):
    """
    If a column contains vectors/lists/arrays per row, reduce each row to one number.
    If scalar, convert directly to numeric.
    """
    non_null = series.dropna()

    if len(non_null) == 0:
        return pd.Series(np.nan, index=series.index)

    example = non_null.iloc[0]

    if is_vector_like(example):
        return series.apply(
            lambda x: reducer(np.asarray(x, dtype=float))
            if is_vector_like(x) else np.nan
        )

    return pd.to_numeric(series, errors="coerce")


def make_triangle_plot(
    table,
    cols_to_plot,
    color_col="gp_kspace_log_band_powers",
    color_reducer=np.nanmax,
    vector_reducer=np.nanmedian,
    log_cols=None,
    figsize_per_panel=2.5,
    s=12,
    alpha=0.8,
):
    """
    Triangle/corner scatter plot.

    Parameters
    ----------
    table : pandas.DataFrame
        Input dataframe.

    cols_to_plot : list[str]
        Columns to include in the triangle plot.

    color_col : str
        Column used for point color. If vector-valued, reduced using color_reducer.

    color_reducer : callable
        Function used to reduce vector-valued color column, e.g. np.nanmax.

    vector_reducer : callable
        Function used to reduce vector-valued plot columns, e.g. np.nanmedian.

    log_cols : list[str] or None
        Columns to plot on log scale.

    figsize_per_panel : float
        Controls total figure size.

    s : float
        Marker size.

    alpha : float
        Marker transparency.
    """

    if log_cols is None:
        log_cols = []

    # Build scalar plotting dataframe
    plot_df = pd.DataFrame(index=table.index)

    for col in cols_to_plot:
        plot_df[col] = reduce_column_for_triangle(
            table[col],
            reducer=vector_reducer
        )

    color_vals = reduce_column_for_triangle(
        table[color_col],
        reducer=color_reducer
    )

    plot_df["color_val"] = color_vals

    # Drop rows with missing values in any plotted quantity
    needed_cols = cols_to_plot + ["color_val"]
    plot_df = plot_df.dropna(subset=needed_cols)

    n = len(cols_to_plot)

    fig, axs = plt.subplots(
        n,
        n,
        figsize=(figsize_per_panel * n, figsize_per_panel * n),
        squeeze=False
    )

    sc = None

    for i, ycol in enumerate(cols_to_plot):
        for j, xcol in enumerate(cols_to_plot):
            ax = axs[i, j]

            # Only lower triangle + diagonal
            if j > i:
                ax.axis("off")
                continue

            if i == j:
                # Diagonal: histogram of each variable
                ax.hist(plot_df[xcol], bins=30, alpha=0.8)
                ax.set_ylabel("count")
            else:
                sc = ax.scatter(
                    plot_df[xcol],
                    plot_df[ycol],
                    c=plot_df["color_val"],
                    s=s,
                    alpha=alpha
                )

            if xcol in log_cols:
                ax.set_xscale("log")

            if ycol in log_cols and i != j:
                ax.set_yscale("log")

            # Clean labels
            if i == n - 1:
                ax.set_xlabel(xcol)
            else:
                ax.set_xticklabels([])

            if j == 0 and i != j:
                ax.set_ylabel(ycol)
            elif i == j:
                ax.set_title(xcol, fontsize=10)
            else:
                ax.set_yticklabels([])

    # Colorbar from scatter plots
    if sc is not None:
        cbar = fig.colorbar(
            sc,
            ax=axs,
            fraction=0.025,
            pad=0.02
        )
        cbar.set_label(f"{color_col} reduced by {color_reducer.__name__}")

    plt.show()

    return fig, axs, plot_df


cols_to_plot = [
    "LC_flux_err",
    "cadence",
    "gp_bic",
    "intrinsic_std",
    #"intrinsic_mad",
    #'intrinsic_rms',
    'age',
    'n_sectors',
    'rn_alpha',
    'gp_kspace_log_low_high_ratio',

]

fig, axs, plot_df = make_triangle_plot(
    table,
    cols_to_plot=cols_to_plot,
    color_col="gp_kspace_log_band_powers",
    color_reducer=np.nanmax,      # max_power
    vector_reducer=np.nanmedian,  # median for vector columns
    log_cols=["LC_flux_err"],
    s=12,
    alpha=0.8
)

In [ ]:
print('hi')

In [ ]:
from gp_pipeline.gp_table_io import load_gp_results
from gp_pipeline.visualize_gp_afterfit import plot_gp_fit, plot_kspace, plot_noise_gp

#table_subset = table[table['name']=='FSR 1769']
#table_subset = table[table['name']=='ASCC 116']
table_subset = table[table['name']=='BSDL2934']


for i in range(0, len(table_subset)):
    row = table_subset.iloc[i]
    result = row["gp_result"]
    if result is not None:
        print(f"n_components={result.n_components}  BIC={result.bic:.1f}  periods={result.periods}")
    else:
        print("No GP result for this row")

plot_gp_fit(table_subset)
plot_kspace(table_subset, log_y=True)
plot_noise_gp(table_subset) 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example hyperparameters for the user's kernel
sigma = 1.0
gamma = 1.0
period = 5.0
long_sigma = 0.35
long_scale = 18.0

# Example hyperparameters for an SHO-like kernel
# A common stationary SHO covariance form is approximately:
# k(tau) = S0 * w0 * Q * exp(-w0*tau/(2Q)) *
#          [cos(eta*w0*tau) + (1/(2*eta*Q))*sin(eta*w0*tau)]
# where eta = sqrt(1 - 1/(4Q^2)), valid for Q > 1/2.
sho_amp = 1.0
sho_period = 5.0
Q = 3.0
w0 = 2 * np.pi / sho_period
eta = np.sqrt(1 - 1 / (4 * Q**2))

tau = np.linspace(0, 30, 1200)

# User kernel components as a function of lag tau = |t_i - t_j|
k_exp_sine = sigma**2 * np.exp(-gamma * np.sin(np.pi * tau / period)**2)

sqrt3 = np.sqrt(3)
k_matern32 = long_sigma**2 * (1 + sqrt3 * tau / long_scale) * np.exp(-sqrt3 * tau / long_scale)

k_user = k_exp_sine + k_matern32

# SHO example kernel, normalized to have k(0)=sho_amp^2
k_sho_raw = np.exp(-w0 * tau / (2 * Q)) * (
    np.cos(eta * w0 * tau)
    + (1 / (2 * eta * Q)) * np.sin(eta * w0 * tau)
)
k_sho = sho_amp**2 * k_sho_raw
Q2=10
k_sho_raw = np.exp(-w0 * tau / (2 * Q2)) * (
    np.cos(eta * w0 * tau)
    + (1 / (2 * eta * Q2)) * np.sin(eta * w0 * tau)
)
k_sho2 = sho_amp**2 * k_sho_raw

plt.figure(figsize=(8, 4.8))
plt.plot(tau, k_user, label="ExpSineSquared + Matern-3/2")
plt.plot(tau, k_exp_sine, linestyle="--", label="ExpSineSquared component")
plt.plot(tau, k_matern32, linestyle=":", label="Matern-3/2 component")
plt.plot(tau, k_sho, label=f"SHO example, Q={Q}")
plt.plot(tau, k_sho2, label=f"SHO example, Q={Q2}")

plt.axhline(0, linewidth=0.8)
plt.xlabel(r"Lag $\tau = |t_i - t_j|$")
plt.ylabel(r"Covariance $k(\tau)$")
plt.title("Example GP kernel comparison")
plt.legend()
plt.tight_layout()

outpath = "/mnt/data/kernel_comparison.png"
#plt.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()

outpath


In [ ]:
row = table[table['name']=='BSDL2934'].iloc[0]
result = row["gp_result"]
result.features["initial_lsp_peak_periods"]

## Step 5 — Save table

In [ ]:
table

In [25]:
# SAVE HERE~~~
# Save full table (including gp_result) as pickle
# table.to_pickle(OUTPUT)
# print(f"Saved {len(table)} rows to {OUTPUT}")

# Save feature table (no gp_result) as FITS — loadable anywhere without custom imports
save_as_pickle(table, OUTPUT+'.pkl')
print(f"Columns: {list(table.columns)}")

# --- Loading examples ---
# Load from FITS (no custom imports needed):
#   df = load_from_fits(FITS_OUTPUT)
#
# Load from pickle (requires gp_pipeline on sys.path):
#   df = load_from_pickle(OUTPUT, module_dir=MODULE_DIR)

[table_io] saved 246 rows → ./data/cluster_table_Aug21_3sectorcombos_synthetic.pkl
Columns: ['name', 'age', 'origin', 'sectors', 'n_sectors', 'cadence', 'LC_t', 'LC_flux', 'LC_flux_err', 'LSP_freq', 'LSP_power', 'LSP_FAP_power', 'LSP_WN_threshold', 'rn_log10_N', 'rn_alpha', 'intrinsic_std', 'intrinsic_rms', 'intrinsic_mad', 'n_bins_used', 'vn_ratio_gap_aware', 'stetson_j_gap_aware', 'mean_median_offset', 'gamma_p', 'sigma_mad', 'excess_var', 'snr', 'gamma_p_sec', 'g1_int_sec', 'g1_valid', 'n_sectors_valid', 'gamma_p_persector', 'g1_int_persector', 'LSP_peak_power', 'LSP_peak_freq', 'LSP_peak_period', 'LSP_peak_snr_rn', 'LSP_peak_period_diff', 'LSP_peak_excess_diff', 'LSP_peak_snr_at_diff', 'gp_result', 'GP_freq', 'GP_PSD', 'GP_PSD_binned', 'GP_PSD_bin_ratios', 'GP_PSD_high_low_ratio', 'gp_n_components', 'gp_bic', 'gp_log_likelihood', 'gp_jitter', 'gp_periods', 'gp_sho_sigmas', 'gp_sho_omegas', 'gp_sho_Qs', 'gp_initial_lsp_peak_periods', 'gp_rn_sigma', 'gp_rn_period', 'gp_rn_Q', 'gp_ksp

In [ ]:
pwd

In [ ]:
table_2 = load_from_fits(FITS_OUTPUT)

In [ ]:
table_2

In [ ]:
import importlib
import table_pipeline.fits_io as fits_io_mod
importlib.reload(fits_io_mod)
from table_pipeline.fits_io import load_from_fits

In [ ]:
#Load from pickle (requires gp_pipeline on sys.path):
table = load_from_fits(FITS_OUTPUT)

In [ ]:
table['gp_periods'].iloc[100]

In [ ]:
table.columns

In [ ]:
table.head()

In [ ]:
table.columns

In [ ]:
row = table.iloc[0]
for col in table.columns:
    try:
        a=len(row[col])
        print('col, type, length, first10:', col, type(row[col]), a, row[col][0:3])
    except:
        print('col, type, entry', col, type(row[col]), row[col])


In [ ]:
print(table.iloc[0])

In [ ]:
save_as_hdf5(table, OUTPUT+'.hdf5')

In [ ]:
table2 = load_from_hdf5(OUTPUT+'.hdf5')
table2

## 5.1 Regenerate PSD's for previous fits after the fix.

In [ ]:
# from gp_pipeline.gp_table_io import recompute_gp_psd_columns
# recompute_gp_psd_columns(table, freq_lim=(0.1, 12.0), n_bins=10)
# save_as_pickle(table, OUTPUT + '.pkl')

## Step 6 — Visualization examples

In [ ]:
from visualization.vis_lc import plot_cluster_lcs
from visualization.vis_periodogram import plot_cluster_lsp
from visualization.vis_statistics import plot_cluster_statistics, plot_clusters_comparison

example_name = table['name'].iloc[0]
print(f"Example cluster: {example_name}")

In [ ]:
# Raw sector LCs (reads directly from FITS file)
plot_cluster_lcs(table, example_name, lc_dir=LC_DIR)

In [ ]:
# LSP by n_sectors group
plot_cluster_lsp(table, example_name)

In [ ]:
# Summary statistics distribution by n_sectors
plot_cluster_statistics(table, example_name,
                        statistics_list=['intrinsic_std', 'intrinsic_rms',
                                         'vn_ratio_gap_aware', 'stetson_j_gap_aware'])

In [ ]:
# Cross-cluster comparison for n_sectors=1
plot_clusters_comparison(table, n_sectors=1, statistic='intrinsic_std')